# Notebook 02 — Matrix Operations, Dot Products, and Linear Scores

## Objective

Implement and verify elementwise operations, dot products, transposition, matrix multiplication, and the linear score `X @ weights + bias`.

**Prerequisites:** Notebook 01: vectors, matrices, shapes, axes, and indexing.

**Working rule:** Complete the starter cells yourself. Verification cells are deliberately placed after each task and will fail until that task is solved. Keep the assertions; catch only failures that the task explicitly asks you to provoke. Use NumPy plus Python's standard library only. After finishing, restart the kernel and run all cells in order.

**Why this matters:** These operations form the arithmetic core of linear models. Correct output shapes are necessary, but numerical agreement and valid input contracts matter too.


## Shared synthetic data

Rows in `X` are transactions; columns are amount, recent transaction count, account age in days, and merchant diversity. These are synthetic examples, not fraud labels. Predict shapes before running later tasks.

In [ ]:
import numpy as np

v1 = np.array([2.0, -1.0, 3.0])
v2 = np.array([4.0, 5.0, -2.0])
A = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
B = np.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]])
X = np.array([[100.0, 1.0, 365.0, 5.0],
              [250.0, 3.0, 730.0, 2.0],
              [80.0, 2.0, 120.0, 8.0]])
weights = np.array([0.01, 0.5, -0.002, 0.2])
bias = 0.1
assert (A.shape, B.shape, X.shape, weights.shape) == ((2, 3), (3, 2), (3, 4), (4,))


## Task 1 — Elementwise operations and shape contracts

### Goal

Separate operations on matching entries from matrix multiplication.

### Required work

1. Use two compatible `(2, 2)` matrices and implement addition, subtraction, elementwise product, multiplication by a scalar, and safe elementwise division.
2. Before running, write the expected shape of each output and calculate one entry by hand.
3. Compare `*` and `@` on the same matrices in words.

### Expected output

Five arrays with shape `(2, 2)` and a short explanation; division must have no zero denominator.

### Acceptance criteria

- [ ] All five outputs are numerical and have shape `(2, 2)`.
- [ ] At least one entry is hand-checked.
- [ ] The distinction between `*` and `@` is explicit.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
P = np.array([[1.0, 2.0], [3.0, 4.0]])
Q = np.array([[5.0, 6.0], [7.0, 8.0]])
# TODO: assign add, subtract, elementwise, scaled, and divided.
add = subtract = elementwise = scaled = divided = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert all(z is not None and np.asarray(z).shape == (2, 2)
           for z in (add, subtract, elementwise, scaled, divided))
assert np.array_equal(add, np.array([[6., 8.], [10., 12.]]))
assert np.array_equal(subtract, P - Q)
assert np.array_equal(elementwise, P * Q)
assert np.array_equal(scaled, 3 * P)
assert np.allclose(divided, P / Q)
assert not np.array_equal(P * Q, P @ Q)


## Task 2 — Dot product from a loop

### Goal

Show how one scalar combines corresponding vector components.

### Required work

1. Calculate `v1 · v2` by hand.
2. Implement `dot_product_loop(a, b)` using a loop, not `np.dot` or `@` inside the function.
3. Reject a length mismatch and any non-1D input with descriptive `ValueError`s.

### Expected output

One scalar, matching the hand calculation and NumPy.

### Acceptance criteria

- [ ] Valid result is `-3.0`.
- [ ] Both invalid input families are tested.
- [ ] The function uses a loop and does not silently broadcast.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def dot_product_loop(a, b):
    # TODO: convert, validate, accumulate, return.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert np.isclose(dot_product_loop(v1, v2), -3.0)
assert np.isclose(dot_product_loop(v1, v2), np.dot(v1, v2))
for left, right in (([1, 2], [1, 2, 3]), (np.ones((1, 3)), v2)):
    try:
        dot_product_loop(left, right)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid vectors must raise ValueError')


## Task 3 — Transpose and compatible dimensions

### Goal

Explain how transposition changes axes and when it enables a valid product.

### Required work

1. Transpose `A` and a non-square `(2, 3)` matrix; predict shapes before inspecting them.
2. Verify double transposition.
3. Identify which of `A @ B`, `A @ B.T`, and `A.T @ B.T` are valid by inspecting inner dimensions; catch invalid cases.

### Expected output

A shape table and assertions for valid and invalid operations.

### Acceptance criteria

- [ ] `A.T.shape == (3, 2)`.
- [ ] Double transpose preserves `A`.
- [ ] Invalid products are explained rather than left as uncaught errors.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: assign A_transposed and compatible_product.
A_transposed = compatible_product = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert A_transposed is not None and A_transposed.shape == (3, 2)
assert np.array_equal(A_transposed.T, A)
assert compatible_product is not None and compatible_product.shape == (2, 2)
assert np.array_equal(compatible_product, A @ B)
try:
    A @ B.T
except ValueError:
    pass
else:
    raise AssertionError('Inner dimensions do not match')


## Task 4 — Matrix multiplication using three loops

### Goal

Implement the sum of products for each output cell.

### Required work

1. Implement `matmul_loops(left, right)` for numerical 2D arrays with compatible inner dimensions.
2. Allocate the output and use three nested loops; explain the row, column, and summation indices.
3. Verify a hand-computed entry of `A @ B` and a second non-square `(3, 2) @ (2, 4)` case.

### Expected output

A `(2, 2)` matrix `[[58, 64], [139, 154]]` for the first case, plus NumPy agreement for the second.

### Acceptance criteria

- [ ] The function does not call NumPy matrix multiplication internally.
- [ ] Both valid cases match NumPy numerically.
- [ ] The first output is checked against fixed numbers.

**Hint:** Hint: The innermost loop must keep earlier terms; replacing the running total with the newest product loses them.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def matmul_loops(left, right):
    # TODO: convert to numerical arrays; validate dimensions and compatibility.
    # TODO: allocate the result and accumulate each sum of products.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
expected = np.array([[58., 64.], [139., 154.]])
assert np.allclose(matmul_loops(A, B), expected)
A2 = np.array([[1., 2.], [3., 4.], [5., 6.]])
B2 = np.array([[2., 0., 1., 3.], [1., 2., 4., 0.]])
result2 = matmul_loops(A2, B2)
assert result2.shape == (3, 4)
assert np.allclose(result2, A2 @ B2)


## Task 5 — Invalid inputs and debugging

### Goal

Make failed shape contracts explicit and diagnose common NumPy mistakes.

### Required work

1. Test `matmul_loops` on 1D input and incompatible inner dimensions; assert `ValueError`.
2. Verify an identity product.
3. Demonstrate a case where `*` broadcasts but is the wrong operation for a matrix product; explain the incorrect output shape.

### Expected output

Caught errors, a preserved identity product, and one corrected broadcast mistake.

### Acceptance criteria

- [ ] Invalid cases cannot silently return an array.
- [ ] Identity product matches the original.
- [ ] The broadcast mistake is explained and corrected.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: choose two invalid pairs and a square identity matrix.
bad_pairs = None
identity = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert bad_pairs is not None and len(bad_pairs) == 2
for left, right in bad_pairs:
    try:
        matmul_loops(left, right)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid matrix pair was accepted')
assert identity is not None and identity.shape == (2, 2)
assert np.allclose(matmul_loops(identity, identity), identity)
wrong = np.ones((2, 1)) * np.ones((1, 3))
assert wrong.shape == (2, 3)  # broadcasting succeeds; it is not a matrix product


## Task 6 — Raw linear scores for transactions

### Goal

Connect a feature matrix and weights to one score per observation.

### Required work

1. Predict shapes for `X`, `weights`, and the output.
2. Compute `X @ weights + bias` and the same calculation with `matmul_loops`, reshaping weights to a column matrix first.
3. Calculate the first score by hand and explain why raw scores are not automatically fraud probabilities.

### Expected output

A 1D array of three matching scores; first score `1.87`.

### Acceptance criteria

- [ ] Both paths agree within tolerance.
- [ ] The output is 1D with length three.
- [ ] Bias is added exactly once per row.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: compute numpy_scores and manual_scores (both 1D).
numpy_scores = manual_scores = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert numpy_scores is not None and manual_scores is not None
assert np.asarray(numpy_scores).shape == (3,)
assert np.asarray(manual_scores).shape == (3,)
assert np.allclose(numpy_scores, manual_scores)
assert np.isclose(numpy_scores[0], 1.87)


## Task 7 — Reusable scoring function and unit debugging

### Goal

Validate `X @ weights + bias` and recognize when feature units change a score.

### Required work

1. Implement `linear_scores(features, coefficients, intercept)` with 2D features, 1D coefficients, matching feature counts, and a scalar intercept.
2. Test one and three observations; reject wrong feature count, 1D features, and non-scalar intercept.
3. Convert account age from days to months in a copy of `X`; compare scores with the original weights, then adjust only that weight to recover the original scores.

### Expected output

Validated 1D scores, three caught invalid cases, and a short explanation of units.

### Acceptance criteria

- [ ] Valid scores match `X @ weights + bias`.
- [ ] Invalid cases raise `ValueError`.
- [ ] Adjusted units recover the original scores numerically.

**Hint:** Hint: If a feature is divided by about 30, what must happen to its coefficient to keep the product unchanged?

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def linear_scores(features, coefficients, intercept):
    # TODO: convert, validate, calculate and return a 1D result.
    raise NotImplementedError

# TODO: create X_months, adjusted_weights, changed_scores, recovered_scores.
X_months = adjusted_weights = changed_scores = recovered_scores = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert np.allclose(linear_scores(X, weights, bias), X @ weights + bias)
assert linear_scores(X[:1], weights, bias).shape == (1,)
for feat, coeff, intercept in ((X, weights[:-1], bias),
                               (X[0], weights, bias),
                               (X, weights, np.array([bias]))):
    try:
        linear_scores(feat, coeff, intercept)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid score input was accepted')
assert all(z is not None for z in (X_months, adjusted_weights, changed_scores, recovered_scores))
assert X_months.shape == X.shape and adjusted_weights.shape == weights.shape
assert not np.allclose(changed_scores, numpy_scores)
assert np.allclose(recovered_scores, numpy_scores)


## Completion check

- [ ] All task cells are implemented and all provided assertions pass.
- [ ] Expected exceptions are caught inside the relevant test cells.
- [ ] **Restart Kernel and Run All** succeeds in notebook order.
- [ ] Each task has a short written explanation in your own words.
- [ ] No real private data, credentials, or large generated files were added.

**Conceptual exit questions**

1. What is the distinction between elementwise multiplication and a matrix product?
2. What do the three indices in loop-based matrix multiplication mean?
3. Why does `X @ weights` return one number per row?
4. Why can changing a feature's unit change a model score without changing the underlying observation?

**Final deliverable:** One reproducible notebook with functions, numerical and invalid-input tests, transaction scores, and explanations; no separate assignment document.
